---
title: "Practice Activity 5-1: JSON Data Format and APIs"
author: "Shiqi Wu"
format:
  html:
    embed-resources: true
    code-overflow: wrap
    include-in-header:
      text: |
        <style>
        .cell-output-display figure {
          width: 100%;
          max-width: 100%;
        }
        .cell-output-display img {
          max-width: 100% !important;
          height: auto !important;
        }
        </style>
---

# Hierarchical Data, the JSON Data Format, and APIs

In [21]:
import pandas as pd
import requests
import time
from getpass import getpass

## Shows Data

First we'll work with the "Girls" shows JSON data from the reading.

In [3]:
# Fetch data from a URL
response = requests.get("https://dlsun.github.io/pods/data/tvshows.json")

data_shows = response.json()

In [4]:
df_shows = pd.json_normalize(data_shows)
df_shows

,id,url,name,type,language,genres,status,runtime,premiered,officialSite,...,externals.thetvdb,externals.imdb,image.medium,image.original,network,webChannel.id,webChannel.name,webChannel.country.name,webChannel.country.code,webChannel.country.timezone
0,139,http://www.tvmaze.com/shows/139/girls,Girls,Scripted,English,"[Drama, Romance]",Ended,30,2012-04-15,http://www.hbo.com/girls,...,220411,tt1723816,http://static.tvmaze.com/uploads/images/medium...,http://static.tvmaze.com/uploads/images/origin...,NaN,NaN,NaN,NaN,NaN,NaN
1,722,http://www.tvmaze.com/shows/722/the-golden-girls,The Golden Girls,Scripted,English,"[Drama, Comedy]",Ended,30,1985-09-14,NaN,...,71292,tt0088526,http://static.tvmaze.com/uploads/images/medium...,http://static.tvmaze.com/uploads/images/origin...,NaN,NaN,NaN,NaN,NaN,NaN
2,23542,http://www.tvmaze.com/shows/23542/good-girls,Good Girls,Scripted,English,"[Drama, Comedy, Crime]",Running,60,2018-02-26,https://www.nbc.com/good-girls?nbc=1,...,328577,tt6474378,http://static.tvmaze.com/uploads/images/medium...,http://static.tvmaze.com/uploads/images/origin...,NaN,NaN,NaN,NaN,NaN,NaN
3,6771,http://www.tvmaze.com/shows/6771/the-powerpuff...,The Powerpuff Girls,Animation,English,"[Comedy, Action, Science-Fiction]",Running,15,2016-04-04,https://www.cartoonnetwork.com/video/powerpuff...,...,307473,tt4718304,http://static.tvmaze.com/uploads/images/medium...,http://static.tvmaze.com/uploads/images/origin...,NaN,NaN,NaN,NaN,NaN,NaN
4,42726,http://www.tvmaze.com/shows/42726/florida-girls,Florida Girls,Scripted,English,[Comedy],Running,30,2019-07-10,https://poptv.com/floridagirls,...,363682,tt8548870,http://static.tvmaze.com/uploads/images/medium...,http://static.tvmaze.com/uploads/images/origin...,NaN,NaN,NaN,NaN,NaN,NaN
5,32087,http://www.tvmaze.com/shows/32087/chicken-girls,Chicken Girls,Scripted,English,"[Drama, Children, Music]",Running,16,2017-09-05,https://www.youtube.com/playlist?list=PLVewHiZ...,...,339854,NaN,http://static.tvmaze.com/uploads/images/medium...,http://static.tvmaze.com/uploads/images/origin...,NaN,274.0,Brat,United States,US,America/New_York
6,33320,http://www.tvmaze.com/shows/33320/derry-girls,Derry Girls,Scripted,English,[Comedy],Running,30,2018-01-04,http://www.channel4.com/programmes/derry-girls,...,338903,tt7120662,http://static.tvmaze.com/uploads/images/medium...,http://static.tvmaze.com/uploads/images/origin...,NaN,NaN,NaN,NaN,NaN,NaN
7,1955,http://www.tvmaze.com/shows/1955/the-powerpuff...,The Powerpuff Girls,Animation,English,"[Action, Children, Crime]",Ended,30,1998-11-18,NaN,...,76200,tt0175058,http://static.tvmaze.com/uploads/images/medium...,http://static.tvmaze.com/uploads/images/origin...,NaN,NaN,NaN,NaN,NaN,NaN
8,1073,http://www.tvmaze.com/shows/1073/bomb-girls,Bomb Girls,Scripted,English,"[Drama, Romance, War]",Ended,60,2012-01-04,NaN,...,254378,tt1955311,http://static.tvmaze.com/uploads/images/medium...,http://static.tvmaze.com/uploads/images/origin...,NaN,NaN,NaN,NaN,NaN,NaN
9,525,http://www.tvmaze.com/shows/525/gilmore-girls,Gilmore Girls,Scripted,English,"[Drama, Comedy, Romance]",Ended,60,2000-10-05,NaN,...,76568,tt0238784,http://static.tvmaze.com/uploads/images/medium...,http://static.tvmaze.com/uploads/images/origin...,NaN,NaN,NaN,NaN,NaN,NaN


1\. Summarize the networks represented by these shows and the number of these shows that aired on each network.

In [5]:
network_counts = df_shows["network.name"].value_counts(dropna=False)

network_counts

network.name
NBC                2
Cartoon Network    2
HBO                1
Pop                1
NaN                1
Channel 4          1
Global             1
The CW             1
Name: count, dtype: int64

2\. Find the number of seasons for each show in the data. Do this two ways: one which uses `df_shows`, and another that first flattens `data_shows` to a different data frame.

In [6]:
season_counts = df_shows[["name"]].copy()

season_counts["Number of seasons"] = df_shows["seasons"].apply(len)

season_counts

,name,Number of seasons
0,Girls,6
1,The Golden Girls,7
2,Good Girls,3
3,The Powerpuff Girls,3
4,Florida Girls,1
5,Chicken Girls,5
6,Derry Girls,2
7,The Powerpuff Girls,6
8,Bomb Girls,2
9,Gilmore Girls,8


In [7]:
df_seasons = pd.json_normalize(
    data_shows,
    record_path="seasons",
    meta=["id", "name"],
    meta_prefix="show."
)

season_counts_flat = (
    df_seasons.groupby(["show.id", "show.name"])
    .size()
    .reset_index(name="Number of seasons")
)

season_counts_flat

,show.id,show.name,Number of seasons
0,139,Girls,6
1,525,Gilmore Girls,8
2,722,The Golden Girls,7
3,1073,Bomb Girls,2
4,1955,The Powerpuff Girls,6
5,6771,The Powerpuff Girls,3
6,23542,Good Girls,3
7,32087,Chicken Girls,5
8,33320,Derry Girls,2
9,42726,Florida Girls,1


3\. For each episode, find the length (number of characters) of the title. Then create summaries to answer: Which show tends to have the longest episode titles? The shortest?

In [8]:
df_episodes = pd.json_normalize(
    data_shows,
    record_path=["seasons", "episodes"],
    meta=["id", "name"],
    meta_prefix="show."
)

df_episodes["Title length"] = df_episodes["name"].str.len()

df_episodes[["show.id", "show.name", "name", "Title length"]].head()

,show.id,show.name,name,Title length
0,139,Girls,Pilot,5
1,139,Girls,Vagina Panic,12
2,139,Girls,All Adventurous Women Do,24
3,139,Girls,Hannah's Diary,14
4,139,Girls,Hard Being Easy,15


In [9]:
title_summary = (
    df_episodes.groupby(["show.id", "show.name"])["Title length"]
    .agg(["count", "mean", "median"])
    .sort_values("mean", ascending=False)
)

title_summary

,,count,mean,median
show.id,show.name,,,
1955,The Powerpuff Girls,82,27.951220,29.0
525,Gilmore Girls,153,21.431373,21.0
42726,Florida Girls,10,20.400000,23.0
722,The Golden Girls,181,19.723757,18.0
6771,The Powerpuff Girls,119,16.521008,14.0
32087,Chicken Girls,76,16.434211,13.0
23542,Good Girls,26,15.461538,14.5
1073,Bomb Girls,19,14.421053,15.0
139,Girls,63,13.111111,11.0


Answer: In this data set, The Powerpuff Girls (ID 1955) tends to have the longest episode titles, with an average of about 27.95 characters. Derry Girls tends to have the shortest, with an average of 12.50 characters. Their median title lengths are 29 and 9 characters, respectively, which supports the same conclusion.

4\. Do any cast members in the data set share a birthday with you? Who, and what show are they on? (If there isn't anyone, try a different day.)

In [10]:
df_cast = pd.json_normalize(
    data_shows,
    record_path="cast",
    meta=["id", "name"],
    meta_prefix="show."
)

df_cast[["person.name", "person.birthday", "show.id", "show.name"]].head()

,person.name,person.birthday,show.id,show.name
0,Lena Dunham,1986-05-13,139,Girls
1,Allison Williams,1988-04-13,139,Girls
2,Jemima Kirke,1985-04-26,139,Girls
3,Zosia Mamet,1988-02-02,139,Girls
4,Adam Driver,1983-11-19,139,Girls


In [11]:
birthdays = pd.to_datetime(df_cast["person.birthday"], errors="coerce")

birthday_matches = df_cast[
    (birthdays.dt.month == 2) & (birthdays.dt.day == 12)
]

birthday_matches = birthday_matches.drop_duplicates(subset=["person.id", "show.id"])

birthday_matches[["person.name", "person.birthday", "show.id", "show.name"]]

,person.name,person.birthday,show.id,show.name
76,Tara Strong,1973-02-12,1955,The Powerpuff Girls


Answer: Yes. Tara Strong shares my birthday, February 12. She is listed in the cast of The Powerpuff Girls (ID 1955) in this data set.

## TVMaze API

Now you will work with the [TVMaze API](http://www.tvmaze.com/api) from the reading. Use the API to request JSON data that you can use to answer the following questions.

1\. What was the longest show that aired in the U.S. on February 4, 2018?

_Hint:_ Use the ["Schedule" endpoint](http://www.tvmaze.com/api#schedule) to first get the data for all shows that aired on that date.

In [12]:
schedule_url = "https://api.tvmaze.com/schedule"

params = {"country": "US","date": "2018-02-04"}

schedule_response = requests.get(schedule_url, params=params, timeout=30)
schedule_response.raise_for_status()

schedule_data = schedule_response.json()

print("Number of episode records:", len(schedule_data))

Number of episode records: 41


In [13]:
df_schedule = pd.json_normalize(schedule_data)

df_schedule[["show.id", "show.name", "name", "airdate", "runtime"]].head()

,show.id,show.name,name,airdate,runtime
0,7795,1st Look,Going for Gold in PyeongChang,2018-02-03,30
1,5599,America's Test Kitchen,Ultimate Italian,2018-02-03,30
2,21928,Marvel's Spider-Man,"Spider-Island, Part One",2018-02-04,30
3,21928,Marvel's Spider-Man,"Spider-Island, Part Two",2018-02-04,30
4,22084,Ben 10,High Stress Express,2018-02-04,11


In [14]:
df_target_day = df_schedule[df_schedule["airdate"] == "2018-02-04"].copy()

max_runtime = df_target_day["runtime"].max()

longest_episodes = df_target_day[df_target_day["runtime"] == max_runtime]

print("Records with missing runtime:", df_target_day["runtime"].isna().sum())

longest_episodes[["show.id", "show.name", "name", "airdate", "runtime"]]

Records with missing runtime: 0


,show.id,show.name,name,airdate,runtime
23,6011,Super Bowl,Super Bowl LII - New England Patriots vs. Phil...,2018-02-04,210


Answer: Among the returned records with an airdate of February 4, 2018, Super Bowl had the longest episode. Super Bowl LII had a runtime of 210 minutes, or 3 hours and 30 minutes.

2\. Among all shows that aired in the U.S. on Feburary 4, 2018, which non-voice actors appeared on more than one show? Note: some people are credited with multiple rows on the same show and thus appear as multiple rows in the data frame.

Hint: You will need to write a for loop to make multiple requests. Use the "show.id" from the data from part 1, and use the ["Shows" endpoint](http://www.tvmaze.com/api#show-cast) to get the cast of each show. Don't forget to stagger your requests, or you will be blocked by the website!

In [15]:
shows_to_query = df_target_day[["show.id", "show.name"]].drop_duplicates(subset="show.id")

print("Number of unique shows:", len(shows_to_query))

shows_to_query.head()

Number of unique shows: 36


,show.id,show.name
2,21928,Marvel's Spider-Man
4,22084,Ben 10
6,8835,Meet the Press
7,15779,CBS News Sunday Morning
8,7249,Xtreme Off-Road


In [16]:
cast_records = []

for show_id in shows_to_query["show.id"]:
    cast_url = f"https://api.tvmaze.com/shows/{show_id}/cast"
    cast_response = requests.get(cast_url, timeout=30)
    cast_response.raise_for_status()

    show_cast = cast_response.json()

    for cast_member in show_cast:
        cast_records.append({
            "show_id": show_id,
            **cast_member
        })

    time.sleep(0.5)

print("Number of cast records:", len(cast_records))

Number of cast records: 846


In [17]:
df_api_cast = pd.json_normalize(cast_records)

non_voice_cast = df_api_cast[df_api_cast["voice"] == False].copy()

actor_show_pairs = non_voice_cast.drop_duplicates(subset=["person.id", "show_id"])

print("Number of unique actor-show pairs:", len(actor_show_pairs))

actor_show_pairs[["person.id", "person.name", "show_id", "voice"]].head()

Number of unique actor-show pairs: 154


,person.id,person.name,show_id,voice
5,179004,Kristen Welker,8835,False
6,65834,Chuck Todd,8835,False
7,108921,Chris Wallace,8835,False
8,214144,Garrick Utley,8835,False
9,214145,Tim Russert,8835,False


In [18]:
actor_show_counts = (
    actor_show_pairs.groupby(["person.id", "person.name"])["show_id"]
    .nunique()
    .reset_index(name="Number of shows")
)

multiple_show_actors = actor_show_counts[
    actor_show_counts["Number of shows"] > 1
].sort_values("Number of shows", ascending=False)

multiple_show_actors

,person.id,person.name,Number of shows
68,102424,John Dickerson,2
113,162299,Tom Llamas,2


In [19]:
matching_cast = actor_show_pairs[
    actor_show_pairs["person.id"].isin(multiple_show_actors["person.id"])
]

actor_show_details = matching_cast.merge(
    shows_to_query,
    left_on="show_id",
    right_on="show.id",
    validate="many_to_one"
)

actor_show_details[["person.name", "show.name"]].sort_values(["person.name", "show.name"])

,person.name,show.name
0,John Dickerson,CBS News Sunday Morning
1,John Dickerson,Face the Nation
2,Tom Llamas,ABC World News Tonight with David Muir
3,Tom Llamas,NBC Nightly News


Answer: Based on the TVMaze show-level cast data, John Dickerson and Tom Llamas are each listed as non-voice cast members of two shows in the selected schedule. John Dickerson is listed for CBS News Sunday Morning and Face the Nation. Tom Llamas is listed for ABC World News Tonight with David Muir and NBC Nightly News. Each person was counted only once per show.

# Tasty API

[Tasty.co](http://tasty.co) is a website and app that offers food recipes. They have made these recipes available through a [a REST API](https://rapidapi.com/apidojo/api/tasty). However, unlike the TVMaze API, this one requires authentication.

Specifically, you will need to create an account and subscribe to the "Basic" (free) plant. You will then be provided with an API key (X-Rapid-API-Key) that will need to be supplied with every request you make. This is used to track and limit usage.

1. [create an account](https://rapidapi.com/apidojo/api/tasty) and subscribe to the "Basic" (free) plan
2. log in and copy the X-RapidAPI-Key, which is a long string of letters and digits
3. paste this key to replace "PUT-YOUR-KEY-HERE" in the `headers` below

If you did everything correctly, then running the cell below should return a JSON object containing all the tags recognized by the Tasty API.

In [28]:
domain = "https://tasty.p.rapidapi.com"
endpoint = "tags/list"
url = f"{domain}/{endpoint}"

api_key = getpass("Enter your RapidAPI key: ").strip()
while not api_key:
    print("No key entered. Paste your key into the hidden input box.")
    api_key = getpass("Enter your RapidAPI key: ").strip()

headers = {
    "X-RapidAPI-Key": api_key,
    "X-RapidAPI-Host": "tasty.p.rapidapi.com"
}

response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()
tags_data = response.json()

print("Status code:", response.status_code)

No key entered. Paste your key into the hidden input box.
Status code: 200


In [29]:
df_tags = pd.json_normalize(tags_data["results"])

df_tags.head()

,root_tag_type,name,id,parent_tag_name,display_name,type
0,cuisine,brazilian,64446,central_south_american,Brazilian,central_south_american
1,cuisine,german,64450,european,German,european
2,healthy,healthy,64466,NaN,Healthy,healthy
3,dietary,vegetarian,64469,dietary,Vegetarian,dietary
4,seasonal,christmas,64473,holidays,Christmas,holidays


You will need to pass these `headers` with every HTTP request to the API. The API key (X-RapidAPI-Key) is how the server keeps track of how many requests you have made.

Take a look at [the documentation](https://rapidapi.com/apidojo/api/tasty). We will use the recipes/list endpoint.

Make sure you are logged into the account you are created, and select the recipes/list endpoint from the menu at left. Notice that this brings up a form that you can fill in, which generates the corresponding code.  By default, it provides Node.js code; change this to Python with the Requests client.

1\. Search for recipes containing "daikon" (an Asian radish) and request the JSON data, and convert it to a Pandas data frame.

In [30]:
recipes_url = "https://tasty.p.rapidapi.com/recipes/list"
daikon_params = {"from": 0, "size": 40, "q": "daikon"}

daikon_response = requests.get(
    recipes_url,
    headers=headers,
    params=daikon_params,
    timeout=30
)
daikon_response.raise_for_status()
daikon_data = daikon_response.json()

print("Total results:", daikon_data["count"])
print("Results received:", len(daikon_data["results"]))

Total results: 16
Results received: 16


In [31]:
df_daikon = pd.json_normalize(daikon_data["results"])

df_daikon[["id", "name", "price.portion"]].head()

,id,name,price.portion
0,6046,Instant Pot Beef Bulgogi,1050.0
1,4490,Top Chef Junior Pork Bánh Mì Burger,650.0
2,5643,Banh Mi Meatball Sandwich,700.0
3,9820,Chicken Bánh Mì,800.0
4,4258,How To Make Vegan Kimchi,650.0


2\. How many recipes containing daikon are there? Which one is the cheapest per portion?

In [32]:
print("Number of results:", len(df_daikon))

[column for column in df_daikon.columns if "price" in column.lower()]

Number of results: 16


['price.total',
 'price.updated_at',
 'price.portion',
 'price.consumption_total',
 'price.consumption_portion']

In [33]:
df_daikon[["id", "name", "price.portion"]].sort_values("price.portion")

,id,name,price.portion
10,4711,Vegan Tofu Bao Buns With Pickled Vegetables,300.0
6,4621,Low-Carb Pad Thai,500.0
1,4490,Top Chef Junior Pork Bánh Mì Burger,650.0
4,4258,How To Make Vegan Kimchi,650.0
2,5643,Banh Mi Meatball Sandwich,700.0
3,9820,Chicken Bánh Mì,800.0
7,6542,Fried Carrot Cake,850.0
9,5152,Grilled Lemongrass Pork Bánh Mì,850.0
11,4326,How To Make Vegan Pho,900.0
14,9225,Burnt End Bánh Mì,1000.0


Answer: The API returned 16 recipes for “daikon.” Vegan Tofu Bao Buns With Pickled Vegetables has the lowest price per portion, with a price.portion value of 300.

3\. Find recipes containing avocado and request the JSON data.

**Hint:** Note that there are hundreds of results, but the API only returns 20 results by default and only 40 results maximum, even if you specify the `size=` parameter, so you will need to use a `for` loop, incrementing the `from=` parameter. Be sure to respect the API's rate limits (or you may be blocked!)

**Note:** Recall from the example in the reading that we created an empty list `episodes = []` and then added the results of each request to it in the for loop with `episodes.extend(response.json())`. But note that in the recipes/list endpoint there are two keys: count and results. We only want the results so try instead `.extend(response.json()).get("results")`.

**Suggestion:** Try writing a loop that only makes 2 or 3 requests first so you can test that it's working correctly. Also, make sure you use separate cells for code that makes requests to the API versus processing of the results; you don't want to rerun the requests unless absolutely necessary!

In [ ]:
avocado_recipes = []

for offset in [0, 40]:
    avocado_params = {"from": offset, "size": 40, "q": "avocado"}

    avocado_response = requests.get(
        recipes_url,
        headers=headers,
        params=avocado_params,
        timeout=30
    )
    avocado_response.raise_for_status()
    avocado_page = avocado_response.json()

    avocado_total = avocado_page["count"]
    avocado_recipes.extend(avocado_page["results"])

    print("Starting position:", offset)
    print("Results received:", len(avocado_page["results"]))
    print("Total results:", avocado_total)

    time.sleep(1)

Starting position: 0
Results received: 40
Total results: 444
Starting position: 40
Results received: 40
Total results: 444


In [ ]:
for offset in range(len(avocado_recipes), avocado_total, 40):
    avocado_params = {"from": offset, "size": 40, "q": "avocado"}

    avocado_response = requests.get(
        recipes_url,
        headers=headers,
        params=avocado_params,
        timeout=30
    )
    avocado_response.raise_for_status()
    avocado_page = avocado_response.json()

    avocado_recipes.extend(avocado_page["results"])

    print("Starting position:", offset)
    print("Results received:", len(avocado_page["results"]))

    time.sleep(1)

print("Total collected:", len(avocado_recipes))

Starting position: 80
Results received: 40
Starting position: 120
Results received: 40
Starting position: 160
Results received: 40
Starting position: 200
Results received: 40
Starting position: 240
Results received: 40
Starting position: 280
Results received: 40
Starting position: 320
Results received: 40
Starting position: 360
Results received: 40
Starting position: 400
Results received: 40
Starting position: 440
Results received: 4
Total collected: 444


In [ ]:
df_avocado = pd.json_normalize(avocado_recipes)

print("Total rows:", len(df_avocado))
print("Unique recipe IDs:", df_avocado["id"].nunique())
print("Duplicate recipe IDs:", df_avocado["id"].duplicated().sum())
print("Missing recipe IDs:", df_avocado["id"].isna().sum())

Total rows: 444
Unique recipe IDs: 432
Duplicate recipe IDs: 12
Missing recipe IDs: 0


In [ ]:
df_avocado = df_avocado.drop_duplicates(subset="id").reset_index(drop=True)

print("Unique recipes collected:", len(df_avocado))
print("Duplicate recipe IDs:", df_avocado["id"].duplicated().sum())

Unique recipes collected: 432
Duplicate recipe IDs: 0


Note: The API reported 444 results. This run returned 444 rows, including 432 unique recipe IDs and 12 duplicate rows. I removed the duplicates so each recipe is counted once. The answers below are based on the recipes collected, which may not include every matching recipe.

4\. For the recipes containing avocado, compute the proportion of reviews that are positive. For the avocado recipes with over 500 reviews, which one has the highest proportion of positive reviews?


In [ ]:
[column for column in df_avocado.columns if "rating" in column.lower()]

['tips_and_ratings_enabled',
 'user_ratings.count_positive',
 'user_ratings.score',
 'user_ratings.count_negative']

In [ ]:
df_avocado["Total reviews"] = (
    df_avocado["user_ratings.count_positive"]
    + df_avocado["user_ratings.count_negative"]
)

df_avocado["Positive proportion"] = (
    df_avocado["user_ratings.count_positive"]
    / df_avocado["Total reviews"].where(df_avocado["Total reviews"] > 0)
)

df_avocado[["name", "Total reviews", "Positive proportion"]].head()

                          name  Total reviews  Positive proportion
0           Avocado Carbonara         2771.0             0.896788
1         Avocado Lime Salmon         2143.0             0.980401
2  Avocado Quinoa Power Salad         2749.0             0.939614
3   Shrimp & Avocado Tostadas          962.0             0.968815
4               Avocado Toast          401.0             0.937656


In [ ]:
popular_recipes = df_avocado[df_avocado["Total reviews"] > 500]

highest_proportion = popular_recipes["Positive proportion"].max()

best_recipes = popular_recipes[
    popular_recipes["Positive proportion"] == highest_proportion
]

best_recipes[["name", "Total reviews", "Positive proportion"]]

                                name  Total reviews  Positive proportion
11  Grilled Salmon With Avocado Salsa         1898.0             0.987355


Answer: Among the avocado recipes collected with more than 500 reviews, Grilled Salmon With Avocado Salsa has the highest proportion of positive reviews. It has 1898 reviews, and approximately 98.74% are positive.

5\. Take the avocado JSON data from above (you do NOT need to read in the data from the REST API again). How many recipes are vegetarian? You should be able to identify this from the "tags" attribute.

Hint: Try using `json_normalize` with "tags" as the record path to flatten the data so that there is one row for each tag.

In [ ]:
df_avocado_tags = pd.json_normalize(
    avocado_recipes,
    record_path="tags",
    meta=["id", "name"],
    meta_prefix="recipe."
)

df_avocado_tags[["recipe.id", "recipe.name", "name"]].head()

,recipe.id,recipe.name,name
0,56,Avocado Carbonara,north_american
1,56,Avocado Carbonara,italian
2,56,Avocado Carbonara,comfort_food
3,56,Avocado Carbonara,healthy
4,56,Avocado Carbonara,easy


In [ ]:
vegetarian_recipes = df_avocado_tags[
    df_avocado_tags["name"] == "vegetarian"
]

vegetarian_count = vegetarian_recipes["recipe.id"].nunique()

print("Number of vegetarian recipes:", vegetarian_count)

Number of vegetarian recipes: 206


Answer: Among the avocado recipes collected, 206 unique recipes have the “vegetarian” tag. Each recipe was counted only once using its recipe ID.